In [1]:
import pandas as pd
import time
pd.set_option('display.max_rows', 100)

In [2]:
start=time.perf_counter()

In [3]:
from DatabaseCreator import DatabaseCreator
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [4]:
stats_current_gameweek = 16
picks_current_gameweek = 16
# last_completed_gameweek = 13
# current_gameweek = 14
# next_gameweek = 15
current_season = 20252026

In [5]:
# Run SELECT and get results
view = db_creator.run_sql("""SELECT *
                        FROM playergw
                          """)
view.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199350 entries, 0 to 199349
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   player_name_id              199350 non-null  object 
 1   element                     182320 non-null  float64
 2   value                       182320 non-null  float64
 3   season                      199350 non-null  float64
 4   event                       199350 non-null  float64
 5   fixture                     182320 non-null  float64
 6   total_points                182320 non-null  float64
 7   minutes                     182320 non-null  float64
 8   goals_scored                182320 non-null  float64
 9   assists                     182320 non-null  float64
 10  team_elo                    199350 non-null  float64
 11  opp_team_elo                199350 non-null  float64
 12  position                    155844 non-null  object 
 13  bonus         

In [7]:
# Run SELECT and get results
view = db_creator.run_sql(f"""SELECT *
                        FROM playergw
                        WHERE season = 20252026
                        AND event = {stats_current_gameweek}
                          """)
view.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 760 entries, 0 to 759
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   player_name_id              760 non-null    object 
 1   element                     760 non-null    float64
 2   value                       760 non-null    float64
 3   season                      760 non-null    float64
 4   event                       760 non-null    float64
 5   fixture                     760 non-null    float64
 6   total_points                760 non-null    float64
 7   minutes                     760 non-null    float64
 8   goals_scored                760 non-null    float64
 9   assists                     760 non-null    float64
 10  team_elo                    760 non-null    float64
 11  opp_team_elo                760 non-null    float64
 12  position                    760 non-null    object 
 13  bonus                       760 non

In [8]:
# Run SELECT and get results
view = db_creator.run_sql("""
SELECT *
FROM (
    SELECT *,
    SUM(minutes) OVER (PARTITION BY player_name_id, season) AS player_season_minutes_total               
    FROM playergw
) subquery
WHERE player_season_minutes_total <1
ORDER BY player_name_id ASC, season ASC, event ASC

""")
view

,player_name_id,element,value,season,event,fixture,total_points,minutes,goals_scored,assists,...,expected_goals_conceded,starts,cbi,defensive_contribution,recoveries,tackles,saves,team_name,opp_team_name,player_season_minutes_total
0,ÃÂngelo Gabriel Borges Damaceno,151.0,45.0,20242025.0,1.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Chelsea,Man City,0.0
1,ÃÂngelo Gabriel Borges Damaceno,151.0,45.0,20242025.0,2.0,20.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Chelsea,Wolves,0.0
2,ÃÂngelo Gabriel Borges Damaceno,151.0,45.0,20242025.0,3.0,23.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Chelsea,Crystal Palace,0.0
3,ÃÂngelo Gabriel Borges Damaceno,151.0,45.0,20242025.0,4.0,32.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Chelsea,Bournemouth,0.0
4,ÃÂngelo Gabriel Borges Damaceno,151.0,45.0,20242025.0,5.0,50.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Chelsea,West Ham,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52011,Zidane Iqbal,551.0,42.0,20222023.0,34.0,339.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Man Utd,Aston Villa,0.0
52012,Zidane Iqbal,551.0,42.0,20222023.0,35.0,349.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Man Utd,West Ham,0.0
52013,Zidane Iqbal,551.0,42.0,20222023.0,36.0,359.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Man Utd,Wolves,0.0
52014,Zidane Iqbal,551.0,42.0,20222023.0,37.0,361.0,0.0,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,Man Utd,Bournemouth,0.0


In [9]:
def create_running_averages(db_creator, 
                           metrics_to_average,
                           window_sizes=[5, 10],
                           include_alltime=True,
                           include_per_90=True,
                           include_raw=True,
                           include_elo_adjusted=True,
                           include_squared_per_90=True,
                           interaction_pairs=None,
                           additional_fields=None):
    """
    Create running averages for specified metrics with optional normalization.
    Stops calculations at the most recent gameweek with total_points data.
    
    Parameters:
    -----------
    include_elo_adjusted : bool, optional
        If True, adjust metrics by opponent ELO difficulty
        Uses ~30% adjustment per 100 ELO difference
    include_squared_per_90 : bool, optional
        If True, include squared per-90 features for capturing non-linear effects
    interaction_pairs : list of tuples, optional
        List of (metric1, metric2) pairs to create interaction features
        Example: [('expected_goals', 'expected_assists'), ('bps', 'minutes')]
        Creates features like metric1_x_metric2_per90_running_avg_prev_{window}
    """
    
    # Base fields that are always included
    base_fields = [
        'player_name_id', 'element', 'season', 'value', 'event', 'minutes', 'total_points',
        'team_elo', 'opp_team_elo', 'position', 'goals_scored', 'bonus',
        'bps', 'clean_sheets', 'goals_conceded', 'was_home',
        'expected_goals', 'expected_assists', 'expected_goal_involvements',
        'expected_goals_conceded','team_name','opp_team_name', 
        'cbi','defensive_contribution','recoveries','tackles',
        'saves'
    ]
    
    if additional_fields:
        base_fields.extend(additional_fields)
    
    select_parts = base_fields.copy()
    
    # Season totals and derived per-90 stats
    select_parts.append('SUM(minutes) OVER (PARTITION BY player_name_id, season) AS player_season_minutes_total')
    select_parts.append("""(SUM(total_points) OVER (PARTITION BY player_name_id, season) / 
                            NULLIF(SUM(minutes) OVER (PARTITION BY player_name_id, season), 0) * 90.0)
                                AS player_season_points_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (total_points / minutes * 90.0) ELSE 0 END AS points_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (expected_goals / minutes * 90.0) ELSE 0 END AS expected_goals_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (expected_assists / minutes * 90.0) ELSE 0 END AS expected_assists_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (expected_goals_conceded / minutes * 90.0) ELSE 0 END AS expected_goals_conceded_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (saves / minutes * 90.0) ELSE 0 END AS saves_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (bps / minutes * 90.0) ELSE 0 END AS bps_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (bonus / minutes * 90.0) ELSE 0 END AS bonus_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (cbi / minutes * 90.0) ELSE 0 END AS cbi_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (defensive_contribution / minutes * 90.0) ELSE 0 END AS defensive_contribution_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (recoveries / minutes * 90.0) ELSE 0 END AS recoveries_per90""")
    select_parts.append("""CASE WHEN minutes > 0 THEN (tackles / minutes * 90.0) ELSE 0 END AS tackles_per90""")

    select_parts.append('team_elo - opp_team_elo AS elo_diff')

    # Interaction features (product of two metrics)
    if interaction_pairs:
        for metric1, metric2 in interaction_pairs:
            # Shorten metric names for interactions
            m1 = metric1.replace('expected_', 'x').replace('_contribution', '_cont')
            m2 = metric2.replace('expected_', 'x').replace('_contribution', '_cont')
            interaction_name = f"{m1}_x_{m2}"
            
            # All-time interaction averages
            if include_alltime:
                if include_per_90:
                    metric1_per90 = f"CASE WHEN minutes > 0 THEN ({metric1} / minutes * 90.0) ELSE 0 END"
                    metric2_per90 = f"CASE WHEN minutes > 0 THEN ({metric2} / minutes * 90.0) ELSE 0 END"
                    interaction_expr = f"({metric1_per90}) * ({metric2_per90})"
                    avg_name = f"{interaction_name}_p90_at"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL 
                                THEN {interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name}""")
                
                # Raw interaction (not per-90)
                if include_raw:
                    raw_interaction_expr = f"{metric1} * {metric2}"
                    avg_name_raw = f"{interaction_name}_at"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL 
                                THEN {raw_interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_raw}""")
            
            # Window-based interaction averages
            for window in window_sizes:
                if include_per_90:
                    metric1_per90 = f"CASE WHEN minutes > 0 THEN ({metric1} / minutes * 90.0) ELSE 0 END"
                    metric2_per90 = f"CASE WHEN minutes > 0 THEN ({metric2} / minutes * 90.0) ELSE 0 END"
                    interaction_expr = f"({metric1_per90}) * ({metric2_per90})"
                    avg_name = f"{interaction_name}_p90_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name}""")
                
                if include_raw:
                    raw_interaction_expr = f"{metric1} * {metric2}"
                    avg_name_raw = f"{interaction_name}_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {raw_interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_raw}""")
                
                # ELO-adjusted interaction
                if include_elo_adjusted and include_per_90:
                    elo_difficulty_multiplier = f"POWER(1.3, (opp_team_elo - team_elo) / 100.0)"
                    metric1_elo_per90 = f"CASE WHEN minutes > 0 THEN (({metric1} / minutes * 90.0) / {elo_difficulty_multiplier}) ELSE 0 END"
                    metric2_elo_per90 = f"CASE WHEN minutes > 0 THEN (({metric2} / minutes * 90.0) / {elo_difficulty_multiplier}) ELSE 0 END"
                    elo_interaction_expr = f"({metric1_elo_per90}) * ({metric2_elo_per90})"
                    avg_name_elo = f"{interaction_name}_elo_p90_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {elo_interaction_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_elo}""")

    # Running average calculations
    for metric in metrics_to_average:
        # Shorten metric name
        short_metric = metric.replace('expected_', 'x').replace('_contribution', '_cont')
        
        # All-time averages
        if include_alltime:
            avg_name = f"{short_metric}_at"
            select_parts.append(f"""
                CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                    AVG(CASE WHEN total_points IS NOT NULL 
                        THEN {metric} ELSE NULL END) OVER (
                        PARTITION BY player_name_id
                        ORDER BY season ASC, event ASC
                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                    )
                END AS {avg_name}""")

            if include_per_90:
                metric_expr = f"CASE WHEN minutes > 0 THEN ({metric} / minutes * 90.0) ELSE 0 END"
                avg_name_per90 = f"{short_metric}_p90_at"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL 
                            THEN {metric_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_per90}""")
                
                # Squared per-90 all-time average
                if include_squared_per_90:
                    squared_expr = f"POWER({metric_expr}, 2)"
                    avg_name_squared = f"{short_metric}_p90sq_at"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL 
                                THEN {squared_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_squared}""")

        for window in window_sizes:
            # Raw running average
            if include_raw:
                avg_name = f"{short_metric}_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {metric} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name}""")
            
            # Per-90 running average
            if include_per_90:
                metric_expr = f"CASE WHEN minutes > 0 THEN ({metric} / minutes * 90.0) ELSE 0 END"
                avg_name_per90 = f"{short_metric}_p90_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {metric_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_per90}""")
                
                # Squared per-90 running average
                if include_squared_per_90:
                    squared_expr = f"POWER({metric_expr}, 2)"
                    avg_name_squared = f"{short_metric}_p90sq_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {squared_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_squared}""")
            
            # ELO-adjusted running average
            if include_elo_adjusted:
                elo_difficulty_multiplier = f"POWER(1.3, (opp_team_elo - team_elo) / 100.0)"
                elo_adjusted_expr = f"{metric} / {elo_difficulty_multiplier}"
                avg_name_elo = f"{short_metric}_elo_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {elo_adjusted_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_elo}""")
                
            # ELO-adjusted per-90
            if include_per_90 and include_elo_adjusted:
                elo_per90_expr = f"CASE WHEN minutes > 0 THEN (({metric} / minutes * 90.0) / {elo_difficulty_multiplier}) ELSE 0 END"
                avg_name_elo_per90 = f"{short_metric}_elo_p90_{window}"
                select_parts.append(f"""
                    CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                        AVG(CASE WHEN total_points IS NOT NULL
                            THEN {elo_per90_expr} ELSE NULL END) OVER (
                            PARTITION BY player_name_id
                            ORDER BY season ASC, event ASC
                            ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                        )
                    END AS {avg_name_elo_per90}""")
                
                # Squared ELO-adjusted per-90
                if include_squared_per_90:
                    squared_elo_expr = f"POWER({elo_per90_expr}, 2)"
                    avg_name_elo_squared = f"{short_metric}_elo_p90sq_{window}"
                    select_parts.append(f"""
                        CASE WHEN (season < 20252026 OR (season = 20252026 AND event <= latest_completed_gw+1)) THEN
                            AVG(CASE WHEN total_points IS NOT NULL
                                THEN {squared_elo_expr} ELSE NULL END) OVER (
                                PARTITION BY player_name_id
                                ORDER BY season ASC, event ASC
                                ROWS BETWEEN {window} PRECEDING AND 1 PRECEDING
                            )
                        END AS {avg_name_elo_squared}""")
    
    select_clause = ',\n                        '.join(select_parts)
    
    # Include CTE for latest completed GW
    query = f"""

        WITH latest_gw AS (
            SELECT 
                MAX(event) AS latest_completed_gw
            FROM playergw
            WHERE season = 20252026 AND total_points IS NOT NULL
        ),
        base AS (
            SELECT 
                playergw.*, 
                latest_gw.latest_completed_gw
            FROM playergw
            CROSS JOIN latest_gw
        )

        SELECT *
        FROM (
            SELECT
                {select_clause}
            FROM base
        ) subquery
        WHERE player_season_minutes_total > 0
        ORDER BY player_name_id ASC, season ASC, event ASC
        """
    
    return db_creator.run_sql(query)


# Example usage
metrics = ['minutes',
 'total_points',
 'team_elo',
 'opp_team_elo',
 'goals_scored',
 'bonus',
 'bps',
 'clean_sheets',
 'goals_conceded',
 'expected_goals',
 'expected_assists',
 'expected_goal_involvements',
 'expected_goals_conceded',
 'cbi',
 'defensive_contribution',
 'recoveries',
 'tackles',
 'saves']

# Define interaction pairs that make sense for FPL
interaction_pairs = [
    ('expected_goal_involvements', 'defensive_contribution'),  # All-round 
]

df = create_running_averages(db_creator, 
                             metrics_to_average=metrics,
                             window_sizes=[3,5,10],
                             include_alltime=True,
                             include_per_90=True,
                             include_raw=True,
                             include_elo_adjusted=True,
                             include_squared_per_90=True,
                             interaction_pairs=interaction_pairs)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147334 entries, 0 to 147333
Columns: 430 entries, player_name_id to saves_elo_p90sq_10
dtypes: float64(425), object(5)
memory usage: 483.3+ MB


In [10]:
import pandas as pd
import numpy as np

current_season = 20252026
current_gw = stats_current_gameweek

# Identify columns to exclude from forward filling (all fpl playergwdf columns) real life data columns
exclude_cols = ['player_name_id', 'season', 'event',
                'fixture', 'total_points', 'minutes',
                'goals_scored', 'assists', 'team_elo', 'opp_team_elo',
                'position', 'bonus', 'bps', 'clean_sheets',
                'goals_conceded', 'was_home', 'expected_assists',
                'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded',
                'starts', 'team_name', 'opp_team_name',
                'cbi','defensive_contribution','recoveries','tackles'
                'saves']

# Create a mask for rows to fill: season 20252026 and event > 10
mask = (df['season'] == current_season) & (df['event'] > current_gw)

# Get columns to fill
cols_to_fill = [col for col in df.columns if col not in exclude_cols]

# Sort by player and event to ensure proper order
df = df.sort_values(['season', 'player_name_id', 'event'])

# Forward fill only for the masked rows
df_filled = df.copy()

for col in cols_to_fill:
    # Forward fill within each player group
    df_filled.loc[mask, col] = df_filled.groupby('player_name_id')[col].ffill()[mask]

In [ ]:
mask = (((df_filled['season'] == 20242025) & (df_filled['event'] > 29)) 
    | ((df_filled['season'] == 20252026) & (df_filled['event'] > 0))
    ) & (df_filled['player_name_id'] == 'Bukayo Saka')

print("Sample of filled data for the requested seasons/events:")
df_filled.loc[mask, ['player_name_id', 'season', 'event', 'minutes',
                     'total_points', 'opp_team_name', 'opp_team_elo',
                     'expected_goals']].head(50)


Sample of filled data for the requested seasons/events:


,player_name_id,season,event,minutes,total_points,opp_team_name,opp_team_elo,expected_goals
21087,Bukayo Saka,20242025.0,21.0,0.0,0.0,Spurs,1047.715211,0.00
21088,Bukayo Saka,20242025.0,22.0,0.0,0.0,Aston Villa,1106.551582,0.00
21089,Bukayo Saka,20242025.0,23.0,0.0,0.0,Wolves,939.894345,0.00
21090,Bukayo Saka,20242025.0,24.0,0.0,0.0,Man City,1234.310334,0.00
21091,Bukayo Saka,20242025.0,25.0,0.0,0.0,Leicester,917.521723,0.00
21092,Bukayo Saka,20242025.0,26.0,0.0,0.0,West Ham,998.298411,0.00
21093,Bukayo Saka,20242025.0,27.0,0.0,0.0,Nott'm Forest,1049.907963,0.00
21094,Bukayo Saka,20242025.0,28.0,0.0,0.0,Man Utd,1048.649909,0.00
21095,Bukayo Saka,20242025.0,29.0,0.0,0.0,Chelsea,1109.512274,0.00
21096,Bukayo Saka,20242025.0,30.0,24.0,8.0,Fulham,1053.766128,0.77


In [12]:
#Add a new staging table & data into the database
table_name = "playergw_transformed"

if __name__ == "__main__":
    db_creator.create_staging_table_then_insert_data(table_name, data=df_filled)
    playergw_transformed = db_creator.table_to_df(table_name=table_name)
    print(playergw_transformed.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147334 entries, 0 to 147333
Columns: 430 entries, player_name_id to saves_elo_p90sq_10
dtypes: float64(425), object(5)
memory usage: 483.3+ MB
None


In [ ]:
finish=time.perf_counter()
print(f'Finished in {round(((finish-start)/60),2)} minute(s)')
#Finished in 27.16 minute(s)

Finished in 27.16 minute(s)
